In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

In [ ]:
os.chdir('/content/drive/MyDrive/Colab Notebooks/AE Safety Dashboard')
print(os.getcwd())
print(os.listdir())

/content/drive/MyDrive/Colab Notebooks/AE Safety Dashboard
['ae.xpt', 'dm.xpt', 'ae_incidence_by_arm.csv', 'ae_severity_by_arm.csv', 'ae_soc_by_arm.csv', 'ae_sae_by_arm.csv', 'prr_results_high_dose.csv', 'safety_signals_flagged.csv', 'Copy of AE SAFETY SIGNAL MONITORING .ipynb', 'AE SAFETY SIGNAL MONITORING .ipynb']


**PART 1A: SETUP — Import libraries and connect to database**

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
# Load .xpt files and push into SQLite
dm = pd.read_sas('dm.xpt', format='xport', encoding='utf-8')
ae = pd.read_sas('ae.xpt', format='xport', encoding='utf-8')

In [ ]:
conn = sqlite3.connect(':memory:')
dm.to_sql('dm', conn, index=False, if_exists='replace')
ae.to_sql('ae', conn, index=False, if_exists='replace')

1191

In [ ]:
print(f"dm table loaded: {dm.shape[0]} subjects, {dm.shape[1]} columns")
print(f"ae table loaded: {ae.shape[0]} AE records, {ae.shape[1]} columns")

dm table loaded: 306 subjects, 25 columns
ae table loaded: 1191 AE records, 35 columns


In [ ]:
print("\nColumn names (raw):")
for c in dm.columns:
    print(repr(c)) # repr() shows hidden whitespace


Column names (raw):
'STUDYID'
'DOMAIN'
'USUBJID'
'SUBJID'
'RFSTDTC'
'RFENDTC'
'RFXSTDTC'
'RFXENDTC'
'RFICDTC'
'RFPENDTC'
'DTHDTC'
'DTHFL'
'SITEID'
'AGE'
'AGEU'
'SEX'
'RACE'
'ETHNIC'
'ARMCD'
'ARM'
'ACTARMCD'
'ACTARM'
'COUNTRY'
'DMDTC'
'DMDY'


In [ ]:
print("Missing value scan across all DM columns ")
for col in dm.columns:
    query = f'''
    SELECT COUNT(*) AS missing
    FROM dm
    WHERE "{col}" IS NULL
    '''
    result = pd.read_sql(query, conn)
    count = result["missing"][0]
    if count > 0:
        pct = round(count / len(dm) * 100, 1)
        print(f"{col}: {count} ")

Missing value scan across all DM columns 
DMDY: 52 


In [ ]:
print("DM columns:")
print(dm.columns.tolist())

print("\nAE columns:")
print(ae.columns.tolist())

DM columns:
['STUDYID', 'DOMAIN', 'USUBJID', 'SUBJID', 'RFSTDTC', 'RFENDTC', 'RFXSTDTC', 'RFXENDTC', 'RFICDTC', 'RFPENDTC', 'DTHDTC', 'DTHFL', 'SITEID', 'AGE', 'AGEU', 'SEX', 'RACE', 'ETHNIC', 'ARMCD', 'ARM', 'ACTARMCD', 'ACTARM', 'COUNTRY', 'DMDTC', 'DMDY']

AE columns:
['STUDYID', 'DOMAIN', 'USUBJID', 'AESEQ', 'AESPID', 'AETERM', 'AELLT', 'AELLTCD', 'AEDECOD', 'AEPTCD', 'AEHLT', 'AEHLTCD', 'AEHLGT', 'AEHLGTCD', 'AEBODSYS', 'AEBDSYCD', 'AESOC', 'AESOCCD', 'AESEV', 'AESER', 'AEACN', 'AEREL', 'AEOUT', 'AESCAN', 'AESCONG', 'AESDISAB', 'AESDTH', 'AESHOSP', 'AESLIFE', 'AESOD', 'AEDTC', 'AESTDTC', 'AEENDTC', 'AESTDY', 'AEENDY']


In [ ]:
# Check which column names are shared between the two tables
# check USUBJID (the join key) is present in both
matching_columns = dm.columns.intersection(ae.columns)

print("Matching columns:")
print(matching_columns.tolist())

Matching columns:
['STUDYID', 'DOMAIN', 'USUBJID']


In [ ]:
#check whether it is actually unique in dm
print(dm['USUBJID'].is_unique)
#see how many AE records each subject has:
print(ae['USUBJID'].value_counts().head(10))


True
USUBJID
01-701-1302    23
01-717-1004    19
01-704-1266    16
01-709-1029    16
01-718-1427    16
01-709-1309    15
01-701-1275    15
01-701-1192    15
01-713-1179    15
01-711-1143    14
Name: count, dtype: int64


**PART 1B: Quick profile before aggregating**

In [ ]:
#Check total subject for each arm
print("Arms actually present in DM ")

pd.read_sql("""
    SELECT ARM, COUNT(*) AS subjects
    FROM dm
    GROUP BY ARM
    ORDER BY ARM
""", conn)

Arms actually present in DM 


,ARM,subjects
0,Placebo,86
1,Screen Failure,52
2,Xanomeline High Dose,84
3,Xanomeline Low Dose,84


Important finding:
1. DM includes a "Screen Failure" arm
2. subjects who were screened but never dosed.
3. Since they never received treatment, they are not relevant when calculating the adverse event rate.


In [ ]:
#Check total ae record for Screen Failure Arm

print("Screen Failure AE check")

pd.read_sql("""
    SELECT d.ARM,COUNT(a.USUBJID) AS ae_record_count
    FROM dm d
    LEFT JOIN ae a
        ON d.USUBJID = a.USUBJID
    WHERE d.ARM = 'Screen Failure'
    GROUP BY d.ARM
""", conn)


Screen Failure AE check


,ARM,ae_record_count
0,Screen Failure,0


Since there is 0 AE records tied to Screen Failure, we can exclude them from query.

**PART 2 : SQL Analyze**




**QUERY 1 — AE incidence rate by treatment arm**

In [ ]:
#For each treatment group,
#find what percentage of people had at least one adverse event (AE)
query_incidence = '''
WITH dosed AS (
    -- only subjects who were actually dosed
    SELECT * FROM dm WHERE ARM != 'Screen Failure'
),

-- Treated people who had at least one AE

subjects_with_ae AS (
    SELECT DISTINCT d.USUBJID, d.ARM
    FROM dosed d
    JOIN ae ON ae.USUBJID = d.USUBJID
)
SELECT
    d.ARM,
    COUNT(DISTINCT d.USUBJID) AS n_subjects,
    COUNT(DISTINCT s.USUBJID) AS subjects_with_ae,
    ROUND(100.0 * COUNT(DISTINCT s.USUBJID) / COUNT(DISTINCT d.USUBJID), 1) AS pct_with_ae
FROM dosed d
LEFT JOIN subjects_with_ae s ON s.USUBJID = d.USUBJID AND s.ARM = d.ARM
GROUP BY d.ARM
ORDER BY pct_with_ae DESC
'''

ae_incidence = pd.read_sql(query_incidence, conn)
print("=== AE incidence rate by arm ===")
ae_incidence

=== AE incidence rate by arm ===


,ARM,n_subjects,subjects_with_ae,pct_with_ae
0,Xanomeline High Dose,84,79,94.0
1,Xanomeline Low Dose,84,77,91.7
2,Placebo,86,69,80.2


Xanomeline High Dose had a higher percentage of subjects experiencing at least one AE

**QUERY 2 — AE severity breakdown by arm**

In [ ]:
#count how many Severe AE events occurred in each treatment arm
query_severity = '''
SELECT
    d.ARM,
    ae.AESEV AS severity,
    COUNT(*) AS ae_count
FROM ae
JOIN dm d ON d.USUBJID = ae.USUBJID
WHERE d.ARM != 'Screen Failure'
GROUP BY d.ARM, ae.AESEV
ORDER BY d.ARM
'''

ae_severity = pd.read_sql(query_severity, conn)
print("\n=== AE severity breakdown by arm ===")
ae_severity



=== AE severity breakdown by arm ===


,ARM,severity,ae_count
0,Placebo,MILD,219
1,Placebo,MODERATE,74
2,Placebo,SEVERE,8
3,Xanomeline High Dose,MILD,306
4,Xanomeline High Dose,MODERATE,139
5,Xanomeline High Dose,SEVERE,10
6,Xanomeline Low Dose,MILD,245
7,Xanomeline Low Dose,MODERATE,165
8,Xanomeline Low Dose,SEVERE,25


Note : Most AEs in all three groups were Mild.
The Low Dose group had the highest number of Severe AE events (25), while High Dose had the most total AE events overall.

**QUERY 3 — Top system organ classes (SOC) by arm**

In [ ]:
query_soc = '''
SELECT
    d.ARM,
    ae.AEBODSYS AS system_organ_class,
    COUNT(*) AS ae_count
FROM ae
JOIN dm d ON d.USUBJID = ae.USUBJID
WHERE d.ARM != 'Screen Failure'
GROUP BY d.ARM, ae.AEBODSYS
ORDER BY ae_count DESC
LIMIT 15
'''

ae_soc = pd.read_sql(query_soc, conn)
print("\n=== Top system organ classes by arm ===")
ae_soc


=== Top system organ classes by arm ===


,ARM,system_organ_class,ae_count
0,Xanomeline High Dose,GENERAL DISORDERS AND ADMINISTRATION SITE COND...,124
1,Xanomeline Low Dose,GENERAL DISORDERS AND ADMINISTRATION SITE COND...,120
2,Xanomeline Low Dose,SKIN AND SUBCUTANEOUS TISSUE DISORDERS,118
3,Xanomeline High Dose,SKIN AND SUBCUTANEOUS TISSUE DISORDERS,111
4,Placebo,GENERAL DISORDERS AND ADMINISTRATION SITE COND...,48
5,Placebo,SKIN AND SUBCUTANEOUS TISSUE DISORDERS,47
6,Xanomeline High Dose,NERVOUS SYSTEM DISORDERS,45
7,Xanomeline Low Dose,NERVOUS SYSTEM DISORDERS,40
8,Xanomeline High Dose,GASTROINTESTINAL DISORDERS,37
9,Placebo,INFECTIONS AND INFESTATIONS,35


Note :
1. The Xanomeline groups had substantially more AE events in several body system categories
2. with General Disorders and Skin Disorders standing out most strongly.

**QUERY 4 — Serious adverse events (SAE) by arm**

In [ ]:

query_sae = '''
SELECT
    d.ARM,
    -- Count serious AEs
    SUM(ae.AESER = 'Y') AS serious_ae_count,

    -- Count all AEs
    COUNT(*) AS total_ae_count,

    -- Calculate percentage that are serious
    ROUND(
        100.0 * SUM(ae.AESER = 'Y') / COUNT(*),
        2
    ) AS pct_serious

FROM ae
JOIN dm d ON d.USUBJID = ae.USUBJID
WHERE d.ARM != 'Screen Failure'
GROUP BY d.ARM
'''

ae_sae = pd.read_sql(query_sae, conn)
print("\nSerious AE count by arm")
ae_sae


=== Serious AE count by arm ===


,ARM,serious_ae_count,total_ae_count,pct_serious
0,Placebo,0,301,0.00
1,Xanomeline High Dose,2,455,0.44
2,Xanomeline Low Dose,1,435,0.23


In [ ]:
ae_incidence.to_csv('ae_incidence_by_arm.csv', index=False)
ae_severity.to_csv('ae_severity_by_arm.csv', index=False)
ae_soc.to_csv('ae_soc_by_arm.csv', index=False)
ae_sae.to_csv('ae_sae_by_arm.csv', index=False)
print("\nSaved CSV files")


Saved CSV files


**PART 2 : R Analyze**

In [ ]:
%load_ext rpy2.ipython

In [ ]:

%%R
library(foreign)

In [ ]:
# Load the data
%%R
dm <- read.xport("dm.xpt")
ae <- read.xport("ae.xpt")

cat("Loaded DM:", nrow(dm), "subjects |  AE:", nrow(ae), "records\n")

Loaded DM: 306 subjects |  AE: 1191 records


In [ ]:
#Remove screen failures
#count how many dosed subjects there are
%%R
dosed <- dm[dm$ARM != "Screen Failure", ]
#Match the AE records to the correct subject using their subject ID
ae_dosed <- merge(ae, dosed[, c("USUBJID", "ARM")], by = "USUBJID")

cat("Dosed subjects:", nrow(dosed), "\n")

Dosed subjects: 254 


PART 1 — Chi-square test: overall AE incidence by arm

In [ ]:
%%R
# Find subjects who had at least one AE
subjects_with_ae <- unique(ae_dosed$USUBJID) #Since One subject can appear many times because they can have multiple AEs.
dosed$HAD_AE <- ifelse(dosed$USUBJID %in% subjects_with_ae, 1, 0)

In [ ]:
%%R
# Create a reusable function to compare one treatment arm with Placebo
compare_arm_to_placebo <- function(arm_name, data) {
  #the treatment arm we're testing + Placebo
  sub <- data[data$ARM %in% c(arm_name, "Placebo"), ]

  tab <- table(sub$ARM, sub$HAD_AE)
  #Chi-square test to check whether AE incidence
  test <- chisq.test(tab)
  cat("\n===", arm_name, "vs Placebo ===\n")
  print(tab)
  cat("Chi-square p-value:", round(test$p.value, 4), "\n")
  return(test$p.value)
}

#Test
p_high <- compare_arm_to_placebo("Xanomeline High Dose", dosed)
p_low  <- compare_arm_to_placebo("Xanomeline Low Dose", dosed)

cat("\n=== Interpretation (alpha = 0.05) ===\n")
cat("High Dose vs Placebo:", ifelse(p_high < 0.05, "STATISTICALLY SIGNIFICANT", "not significant"),
    "(p =", round(p_high, 4), ")\n")
cat("Low Dose vs Placebo:", ifelse(p_low < 0.05, "STATISTICALLY SIGNIFICANT", "not significant"),
    "(p =", round(p_low, 4), ")\n")


=== Xanomeline High Dose vs Placebo ===
                      
                        0  1
  Placebo              17 69
  Xanomeline High Dose  5 79
Chi-square p-value: 0.0141 

=== Xanomeline Low Dose vs Placebo ===
                     
                       0  1
  Placebo             17 69
  Xanomeline Low Dose  7 77
Chi-square p-value: 0.0548 

=== Interpretation (alpha = 0.05) ===
High Dose vs Placebo: STATISTICALLY SIGNIFICANT (p = 0.0141 )
Low Dose vs Placebo: not significant (p = 0.0548 )


**PART 2 — PRR (Proportional Reporting Ratio) disproportionality
analysis**

In [ ]:
%%R
calc_prr <- function(term, test_arm, comparator_arm, data) {
    #Keep only records belonging to the treatment we're testing.
  test_data <- data[data$ARM == test_arm, ]
  #Keep only the comparator group.
  comp_data <- data[data$ARM == comparator_arm, ]

  a <- sum(test_data$AETERM == term)
  b <- sum(test_data$AETERM != term)
  c <- sum(comp_data$AETERM == term)
  d <- sum(comp_data$AETERM != term)

#Calculate PRR
  prr <- (a / (a + b)) / (c / (c + d))
  #Create a result table
 data.frame(
    AETERM = term,
    n_test_arm = a,
    n_placebo = c,
    PRR = round(prr, 2)
)
}

**PART 2B :Find the 15 most frequently reported AE terms overall**

In [ ]:
%%R
#Find the 15 most frequently reported AE terms.
top_terms <- names(
    sort(
        table(ae_dosed$AETERM),
        decreasing = TRUE
    )
)[1:15]

#Calculate PRR for each of those 15 AEs
prr_results <- do.call(rbind, lapply(top_terms, function(t) {
    calc_prr(
        t,
        "Xanomeline High Dose",
        "Placebo",
        ae_dosed
    )
}))
#Sort by PRR
prr_results <- prr_results[order(-prr_results$PRR), ]

cat("\n=== PRR screening: Xanomeline High Dose vs Placebo===\n")
print(prr_results, row.names = FALSE)

# Find potential safety signals
signals <- prr_results[
    prr_results$PRR > 2 &
    prr_results$n_test_arm >= 3,
]

cat("\n=== Potential safety signals (PRR > 2, n >= 3 cases) ===\n")
print(signals, row.names = FALSE)


=== PRR screening: Xanomeline High Dose vs Placebo (top 15 AE terms) ===
                      AETERM n_test_arm n_placebo  PRR
   APPLICATION SITE ERYTHEMA         23         3 5.07
                   DIZZINESS         18         3 3.97
           SINUS BRADYCARDIA         12         2 3.97
                      NAUSEA         13         3 2.87
   APPLICATION SITE PRURITUS         35        10 2.32
                    PRURITUS         38        11 2.29
 APPLICATION SITE IRRITATION         16         7 1.51
                        RASH         18         9 1.32
             SKIN IRRITATION          8         4 1.32
             NASOPHARYNGITIS          8         4 1.32
                       COUGH          7         4 1.16
                    ERYTHEMA         22        13 1.12
 APPLICATION SITE DERMATITIS         12         9 0.88
                    HEADACHE          9         8 0.74
                   DIARRHOEA          4        10 0.26

=== Potential safety signals (PRR > 2, n >= 3

These AEs were reported proportionally more often in High Dose than Placebo

In [ ]:
# Step 3: Save outputs for the dashboard
%%R
write.csv(prr_results, "prr_results_high_dose.csv", row.names = FALSE)
write.csv(signals, "safety_signals_flagged.csv", row.names = FALSE)

cat("\nSaved: prr_results_high_dose.csv, safety_signals_flagged.csv\n")


Saved: prr_results_high_dose.csv, safety_signals_flagged.csv
